# 1. Imports and Data Loading

In [ ]:
import math
import os
# Suppress OpenMP conflicting library warning that causes crashes
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import torch
import seaborn as sns
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import optuna
import gc
from xgboost import XGBClassifier
from xgboost import plot_importance
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report,accuracy_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer

This notebook is designed to work on kaggle and locally, therefore the files must be retrieved.

In [ ]:
def load_data():
    '''try to load the data from the local directory, if not found, load it from the Kaggle dataset. Output the three dataframes: train, test, and submission.'''
    
    try:
        # check if the data files are in kaggle /kaggle/input/competitions/playground-series-s6e6 directory
        print("Trying to load data from kaggle input directory...")
        train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv')
        test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')
        submission = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv')
        print("Data loaded successfully from kaggle input directory.")
    except FileNotFoundError:
        print("Data files not found in kaggle input directory. \nTrying to load data from local directory...")
        # Load data from local directory 
        data_dir = os.getcwd()+ '/data/'
        train = pd.read_csv(data_dir + 'train.csv')
        test = pd.read_csv(data_dir + 'test.csv')
        submission = pd.read_csv(data_dir + 'sample_submission.csv') 
        print("Data loaded successfully from local directory.")

    return train, test, submission


train_df, test_df, submission_df = load_data()

# Check GPU availability
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}. Setting global device to 'cuda'.")
    GLOBAL_DEVICE = 'cuda'
else:
    print("No GPU detected. Setting global device to 'cpu'.")
    GLOBAL_DEVICE = 'cpu'


# 2. Data Exploration

In [ ]:
train_df.describe()

In [ ]:
train_df.notnull().sum()    

In [ ]:
def quality_check(df):
    '''check for missing values, duplicated rows, and data types of each column in the dataframe.
    Parameters:
    df (pandas dataframe): the dataframe to check for quality issues.
    '''
    temp_df = df.copy()

    missing_msg = f"The total number of missing values in the training dataset is: {temp_df.isnull().sum().sum()}"
    print(missing_msg)
    print("-" * len(missing_msg))
    dup_msg = f"The total number of duplicated rows in the training dataset is: {temp_df.duplicated().sum()}"
    print(dup_msg)
    print("-" * len(dup_msg))
    print(f"The data types of each column in the training dataset are:\n{temp_df.dtypes}")
    

quality_check(train_df)

In [ ]:
def class_imbalance_check(df, target_col):
    '''Check for class imbalance in the target column of the dataframe.Accepts the dataframe and the name of the targer column as arguments.'''
    class_counts = df[target_col].value_counts()
    print(f"The class distribution in the target column '{target_col}' is:\n{class_counts}")
    #plot the class distribution as a bar chart using matplotlib
    #give the bars different colors
    #
    plt.figure(figsize=(8, 6))
    class_counts.plot(kind='bar', color=['skyblue', 'lightgreen', 'lightcoral', 'gold'])
    plt.title(f"Class Distribution in the column'{target_col}'")
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.show()

target_column = 'class'
class_imbalance_check(train_df, target_column)

# 3.Data Preprocessing

## 3.1 Categorical feature preprocesssing

The data is prepared for training by dropping unnecessary columns like `id` which will not influence the target class. Furthermore, the target column `class` is mapped to integers for convinience working with models. 
Since `spectral_type` on has 2 unique variables, binary encoding is suitable. In contrast `galaxy_population` has multiple variables and therefore one hot encoding is required to avoid models misinterpreting  the interger representation of the column for varrying magnitudes.  

In [ ]:
def drop_id_column(df):
    '''Drop the 'id' column from the dataframe if it exists.'''
    temp_df = df.copy()
    if 'id' in temp_df.columns:
        temp_df = temp_df.drop(columns=['id'])
        print("The 'id' column has been dropped from the dataframe.")
    else:
        print("The 'id' column does not exist in the dataframe.")
    return temp_df

no_id_train_df = drop_id_column(train_df)
no_id_train_df.head()


In [ ]:
def preprocess_categorical_features(df, target_col):
    '''Preprocess the categorical features in the dataframe. 
    This function will re-map the target variable to numerical values using a dictionary mapping GALAXY to 0, QSO to 1 and STAR to 2. 
    It will also  binary encode the galaxy_population column, and apply one-hot encoding to the spectral_type column.'''

    temp_df = df.copy()
    #Re-map the target variable to numerical values using a dictionary mapping GALAXY to 0, QSO to 1 and STAR to 2.
    class_mapping = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
    temp_df[target_col] = temp_df[target_col].map(class_mapping)

    # Binary encode galaxy_population using a fixed mapping
    galaxy_population_mapping = {'Red_Sequence': 0, 'Blue_Cloud': 1}
    temp_df['galaxy_population'] = temp_df['galaxy_population'].map(galaxy_population_mapping)

    #Applying one-hot encoding to spectral_type column
    temp_df = pd.get_dummies(temp_df, columns=['spectral_type'], drop_first=True, dtype=int)
    return temp_df

target_col = 'class'
preprocessed_categorical_train_df = preprocess_categorical_features(no_id_train_df, target_col)
preprocessed_categorical_train_df.head()

## 3.2 Numerical feature preprocessing

In [ ]:
def plot_numerical_distributions(df, bins=50, sample_size=None):
    """
    Plots the distribution of specified numerical columns in a grid.
    
    Parameters:
    - df: pandas DataFrame
    - bins: int, number of histogram bins
    - sample_size: int, optional (randomly sample data for faster plotting on huge datasets)
    """
    temp_df = df.copy()
    numerical_columns = temp_df.select_dtypes(include = ['int64', 'float64']).columns.tolist()
    numerical_columns = temp_df.select_dtypes(include = ['int64', 'float64']).columns.tolist()
    if sample_size and len(temp_df) > sample_size:
        data_to_plot = df.sample(n=sample_size, random_state=42)
    else:
        data_to_plot = temp_df
        
    n_cols = 3
    n_rows = math.ceil(len(numerical_columns) / n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
    axes = axes.flatten() # Flatten to 1D array for easy iteration
    
    for i, col in enumerate(numerical_columns):
        # Plot histogram with density line
        sns.histplot(data_to_plot[col].dropna(), kde=True, ax=axes[i], 
                     bins=bins, color='skyblue', edgecolor='black')
        
        axes[i].set_title(f'Distribution of {col}', fontsize=14)
        axes[i].set_xlabel(col, fontsize=12)
        axes[i].set_ylabel('Frequency', fontsize=12)
        
    # Remove any empty subplots if the number of columns isn't a multiple of 3
    for j in range(len(numerical_columns), len(axes)):
        fig.delaxes(axes[j])
        
    plt.tight_layout()
    plt.show()

plot_numerical_distributions(preprocessed_categorical_train_df)

In [ ]:
def plot_numerical_boxplots(df, sample_size=None):
    """
    Plots box plots of specified numerical columns in a grid.
    
    Parameters:
    - df: pandas DataFrame
    - sample_size: int, optional (randomly sample data for faster plotting)
    """
    temp_df = df.copy()
    numerical_columns = no_id_train_df.select_dtypes(include = ['int64', 'float64']).columns.tolist()
    if sample_size and len(temp_df) > sample_size:
        data_to_plot = temp_df.sample(n=sample_size, random_state=42)
    else:
        data_to_plot = temp_df
        
    n_cols = 3
    n_rows = math.ceil(len(numerical_columns) / n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
    axes = axes.flatten() # Flatten to 1D array for easy iteration
    
    for i, col in enumerate(numerical_columns):
        # Plot boxplot, styling the outlier dots slightly transparent
        sns.boxplot(x=data_to_plot[col].dropna(), ax=axes[i], color='lightgreen', 
                    flierprops=dict(marker='o', markersize=3, alpha=0.5))
        
        axes[i].set_title(f'Box Plot of {col}', fontsize=14)
        axes[i].set_xlabel(col, fontsize=12)
        
    # Remove any empty subplots if the number of columns isn't a multiple of 3
    for j in range(len(numerical_columns), len(axes)):
        fig.delaxes(axes[j])
        
    plt.tight_layout()
    plt.show()

plot_numerical_boxplots(preprocessed_categorical_train_df)

1. Coordinates: `alpha` and `delta`

**Scaler**: MinMaxScaler

**Why**: These are bounded spatial angles (Right Ascension and Declination). MinMaxScaler will compress them into a tight [0, 1] range while perfectly preserving the exact shape of their uniform/slightly skewed distributions. I don't want to standardize these because they don't have a true "center" (mean) in the traditional statistical sense.

2. The "Well-Behaved" Bands: `g`, `r`, and `i`

**Scaler**: StandardScaler

**Why**: As seen in the distributions, these three photometric bands are mostly bell-shaped (normal distributions) with relatively manageable outliers. StandardScaler will center them at a mean of 0 and a standard deviation of 1, which is exactly what Neural Networks and linear models expect.

3. The "Outlier-Heavy" Bands: `u` and `z`

**Scaler**: RobustScaler

**Why**: The box plots showed that u and z have a tight core but extreme outliers stretching far to the edges. If I use a StandardScaler, those massive outliers will skew the mean and standard deviation, crushing the rest of your normal data into a tiny range. RobustScaler ignores the outliers by scaling based on the Median and the Interquartile Range (IQR).

4. The Skewed Outlier: `redshift`

**Scaler**: QuantileTransformer (or RobustScaler)

**Why**: redshift is the trickiest feature. It has a massive spike near 0 and a long tail of extremely important outliers (which represent Quasars). A QuantileTransformer will smooth out this extreme skew by mapping the data to a normal or uniform distribution, preventing the massive 6.0+ outliers from overwhelming distance-based algorithms, while keeping them distinct from the 0.0 values.

In [ ]:
def fit_scalers(df):
    '''Fits numerical scalers on the provided dataframe and returns them in a dictionary.'''
    scalers = {
        'minmax': MinMaxScaler().fit(df[['alpha', 'delta']]),
        'standard': StandardScaler().fit(df[['g', 'r', 'i']]),
        'robust': RobustScaler().fit(df[['u', 'z']]),
        'quantile': QuantileTransformer().fit(df[['redshift']])
    }
    return scalers

def transform_numerical_features(df, scalers):
    '''Transforms numerical features using pre-fitted scalers to prevent data leakage.'''
    temp_df = df.copy()
    temp_df[['alpha', 'delta']] = scalers['minmax'].transform(temp_df[['alpha', 'delta']])
    temp_df[['g', 'r', 'i']] = scalers['standard'].transform(temp_df[['g', 'r', 'i']])
    temp_df[['u', 'z']] = scalers['robust'].transform(temp_df[['u', 'z']])
    temp_df[['redshift']] = scalers['quantile'].transform(temp_df[['redshift']])
    return temp_df

# For EDA plotting purposes, we scale the entire dataset here.
# NOTE: To prevent data leakage, we will re-fit scalers strictly on the training split later!
eda_scalers = fit_scalers(preprocessed_categorical_train_df)
preprocessed_numerical_train_df = transform_numerical_features(preprocessed_categorical_train_df, eda_scalers)
preprocessed_numerical_train_df.head()


In [ ]:
plot_numerical_distributions(preprocessed_numerical_train_df)

In [ ]:
plot_numerical_boxplots(preprocessed_numerical_train_df)

In [ ]:
preprocessed_numerical_train_df.head()

# 4 Training the model

## 4.1 Base data training
In this iteration I use the preprocessed features. This will allow me to establish a base accuracy score and determine which preprocessing steps will imporve accuracy

In [ ]:
def prepare_training_data(df, target_col):
    '''Splits the dataframe into training and testing datasets.'''
    X = df.drop(columns=[target_col])
    y = df[target_col]
    return train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:


def train_and_evaluate_models(models_dict, X_train, X_test, y_train, y_test, prefix=""):
    '''
    Trains multiple models, evaluates them using classification reports, 
    and returns a DataFrame comparing their performance.
    '''
    results = []
    trained_models = {}
    
    for name, model_instance in models_dict.items():
        model_name = f"{prefix}{name}"
        print(f"--- Training {model_name} ---")
        
        current_model = clone(model_instance)
        current_model.fit(X_train, y_train)
        trained_models[model_name] = current_model
        
        # Make predictions
        y_pred = current_model.predict(X_test)
        
        # Evaluate
        acc = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred, output_dict=True)
        
        print(f"\nClassification Report for {model_name}:")
        print(classification_report(y_test, y_pred))
        print("=" * 60 + "\n")
        
        results.append({
            'Model': model_name,
            'Accuracy': acc,
            'Macro F1-Score': report['macro avg']['f1-score'],
            'Weighted F1-Score': report['weighted avg']['f1-score']
        })
        
    # Create comparison DataFrame
    comparison_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
    return comparison_df, trained_models

# 1. Prepare Data
# We split BEFORE scaling to strictly prevent data leakage!
X_train_raw, X_test_raw, y_train, y_test = prepare_training_data(preprocessed_categorical_train_df, 'class')

# Fit scalers ONLY on training data, then transform both train and test sets
training_scalers = fit_scalers(X_train_raw)
X_train = transform_numerical_features(X_train_raw, training_scalers)
X_test = transform_numerical_features(X_test_raw, training_scalers)

# 2. Define Models
models_to_train = {
    'XGBoost': XGBClassifier(n_estimators=200, random_state=42, n_jobs=-1, tree_method='hist', device=GLOBAL_DEVICE),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'HistGradientBoosting': HistGradientBoostingClassifier(random_state=42),
    'Neural Network (MLP)': MLPClassifier(random_state=42, max_iter=300),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
}

# 3. Train, Evaluate and Compare
comparison_df, trained_models = train_and_evaluate_models(models_to_train, X_train, X_test, y_train, y_test, prefix="base_")

# Display the comparison table
comparison_df

## 4.2 Addressing class imbalance training

As seen earlier the target column is unevenly distributed. There are a couple of ways to balance it. This section will explore them and anaylse their effect on the different models

### 4.2.1 SMOTE Oversampling

In [ ]:
def balance_classes(X, y, method='smote'):
    '''
    Balances the class distribution of the dataset.
    
    Parameters:
    - X: pandas DataFrame, feature variables
    - y: pandas Series, target variable
    - method: str, 'smote' for oversampling or 'under' for undersampling
    
    Returns:
    - X_resampled, y_resampled: DataFrames with balanced classes
    '''
    print(f"Original class distribution:\n{y.value_counts()}\n")
    
    if method == 'smote':
        print("Applying SMOTE (Oversampling)...")
        sampler = SMOTE(random_state=42)
    elif method == 'under':
        print("Applying RandomUnderSampler (Undersampling)...")
        sampler = RandomUnderSampler(random_state=42)
    else:
        raise ValueError("Method must be 'smote' or 'under'")
        
    X_resampled, y_resampled = sampler.fit_resample(X, y)
    print(f"\nClass distribution after {method}:\n{y_resampled.value_counts()}")
    
    return X_resampled, y_resampled


In [ ]:
# Apply SMOTE to balance the training data
X_train_bal, y_train_bal = balance_classes(X_train, y_train, method='smote')

# Re-evaluate the models using the balanced training data
# We reuse the models_to_train dictionary defined in section 4.1
comparison_df_bal, trained_models_bal = train_and_evaluate_models(
    models_to_train, X_train_bal, X_test, y_train_bal, y_test, prefix="smote_"
)

# Combine all trained models
all_trained_models = {**trained_models, **trained_models_bal}

# Combine all comparison dataframes
all_comparisons_df = pd.concat([comparison_df, comparison_df_bal], ignore_index=True)
all_comparisons_df = all_comparisons_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

# Display the comparison table for the balanced approach
all_comparisons_df


**SMOTE Created Noise and Blurred Decision Boundaries**

From the classification reports for Class 2 (STAR):

Base XGBoost: Precision = 0.93, Recall = 0.93
SMOTE XGBoost: Precision = 0.86, Recall = 0.96

SMOTE interpolates between existing points to create new, synthetic data. Because it forced the creation of ~235,000 fake stars to match the galaxy count, it likely generated data points in overlapping regions (the "boundaries" between galaxies and stars). As a result, the model became hyper-sensitive to predicting Class 2 (Recall went up to 0.96), but it started misclassifying a lot of non-stars as stars (Precision plummeted to 0.86).

### 4.2.2 RandomUnderSampler

In [ ]:
# Apply RandomUnderSampler to balance the training data
X_train_under, y_train_under = balance_classes(X_train, y_train, method='under')

# Re-evaluate the models using the undersampled training data
comparison_df_under, trained_models_under = train_and_evaluate_models(
    models_to_train, X_train_under, X_test, y_train_under, y_test, prefix="under_"
)

# Combine all trained models
all_trained_models.update(trained_models_under)

# Combine all comparison dataframes
all_comparisons_df = pd.concat([all_comparisons_df, comparison_df_under], ignore_index=True)
all_comparisons_df = all_comparisons_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

# Display the final comparison table including undersampling
all_comparisons_df


**Undersampling Threw Away Valuable Information**

With Random Undersampling, Class 0 was reduced (GALAXY) from ~300,000 samples down to ~66,000. While this perfectly balanced the dataset, it essentially threw away roughly 230,000 perfectly valid, real-world examples of galaxies! Depriving the model of this much real data made it less confident overall, which is probably why the overall accuracy dropped.

### 4.2.3 Conclusion: Sticking with Base Data

As demonstrated by the results, addressing class imbalance was not fruitful for this dataset. The minority class (`STAR`) is already abundant enough (~66,000 samples) for the models to learn effectively. 

Moving forward, **we will exclusively use the base, un-resampled data** for all subsequent steps.

## 4.3 Feature Engineering

Before combining features or creating new ones, let's analyze the current feature space. We can plot a correlation matrix to understand the linear relationships and interactions between our variables. Then, we can leverage our best performing tree-based model (XGBoost) to plot feature importances.

In [ ]:
# 1. Feature Correlation Matrix
plt.figure(figsize=(12, 8))
correlation_matrix = preprocessed_numerical_train_df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Matrix", fontsize=16)
plt.show()

# Print text output so the LLM can read the matrix
print("\n--- Correlation Matrix ---")
print(correlation_matrix.to_string())

In [ ]:
# 2. XGBoost Feature Importance
# Retrieve our trained base XGBoost model
best_xgb = all_trained_models['base_XGBoost']

fig, ax = plt.subplots(figsize=(10, 6))
plot_importance(best_xgb, ax=ax, importance_type='gain', title="XGBoost Feature Importance (Gain)", grid=False)
plt.show()

# Print text output so the LLM can read the feature importances
print("\n--- XGBoost Feature Importances (Gain) ---")
importances = best_xgb.get_booster().get_score(importance_type='gain')
for feature, gain in sorted(importances.items(), key=lambda x: x[1], reverse=True):
    print(f"{feature}: {gain:.4f}")

# Note: 'gain' represents the relative contribution of the feature to the model, 
# calculated by taking each feature's contribution for each tree in the model.

### 4.3.1 Removing Noisy Features (Top 3 Features Only)

To test if the models are overfitting to noise from the less important features, let's train our base models exclusively on the top 3 most influential features according to XGBoost: `spectral_type_M`, `redshift`, and `galaxy_population`.

In [ ]:
# Select only the top 3 most influential features
top_features = ['spectral_type_M', 'redshift', 'galaxy_population']
X_train_reduced = X_train[top_features]
X_test_reduced = X_test[top_features]

# Re-evaluate the models using the reduced feature space
comparison_df_reduced, trained_models_reduced = train_and_evaluate_models(
    models_to_train, X_train_reduced, X_test_reduced, y_train, y_test, prefix="reduced_"
)

# Combine all trained models
all_trained_models.update(trained_models_reduced)

# Combine all comparison dataframes
all_comparisons_df = pd.concat([all_comparisons_df, comparison_df_reduced], ignore_index=True)
all_comparisons_df = all_comparisons_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

# Display the final comparison table including reduced features
all_comparisons_df


### 4.3.2 Conclusion: Feature Reduction Impact

Reducing the dataset to only the top 3 features caused a significant drop in performance. The `reduced_XGBoost` model's accuracy fell from **~96.8% to ~88.8%**.

**What this means:** The discarded features, specifically the raw photometric bands (`u`, `g`, `r`, `i`, `z`), clearly contain essential predictive signal despite their high multicollinearity and lower individual importance scores. The tree ensembles were successfully extracting valuable patterns from them.

**Next Steps:** We will retain the full base feature set. Instead of removing the highly correlated photometric bands, we will explicitly engineer **Photometric Color Indices** (`u-g`, `g-r`, `r-i`, `i-z`) to help the model directly interpret the color signatures of these objects!

### 4.3.3 Engineering Photometric Color Indices

Now we will explicitly compute the differences between adjacent bands to represent the color and temperature of the objects. We will append these new features to the dataset and scale them appropriately using `RobustScaler` before evaluating their impact on model performance.

In [ ]:
def engineer_color_indices(df):
    '''Computes Photometric Color Indices from the raw bands.'''
    temp_df = df.copy()
    temp_df['u-g'] = temp_df['u'] - temp_df['g']
    temp_df['g-r'] = temp_df['g'] - temp_df['r']
    temp_df['r-i'] = temp_df['r'] - temp_df['i']
    temp_df['i-z'] = temp_df['i'] - temp_df['z']
    return temp_df

def fit_eng_scalers(df):
    '''Extends the base scalers to include the new engineered features.'''
    scalers = fit_scalers(df)
    scalers['robust_colors'] = RobustScaler().fit(df[['u-g', 'g-r', 'r-i', 'i-z']])
    return scalers

def transform_eng_numerical_features(df, scalers):
    '''Transforms both base and engineered features.'''
    temp_df = transform_numerical_features(df, scalers)
    temp_df[['u-g', 'g-r', 'r-i', 'i-z']] = scalers['robust_colors'].transform(temp_df[['u-g', 'g-r', 'r-i', 'i-z']])
    return temp_df

# 1. Engineer features on the raw training and testing splits
X_train_eng_raw = engineer_color_indices(X_train_raw)
X_test_eng_raw = engineer_color_indices(X_test_raw)

# 2. Fit new scalers on the training set and transform both
training_eng_scalers = fit_eng_scalers(X_train_eng_raw)
X_train_eng = transform_eng_numerical_features(X_train_eng_raw, training_eng_scalers)
X_test_eng = transform_eng_numerical_features(X_test_eng_raw, training_eng_scalers)

# 3. Train and Evaluate Models with the new features
comparison_df_eng, trained_models_eng = train_and_evaluate_models(
    models_to_train, X_train_eng, X_test_eng, y_train, y_test, prefix="eng_"
)

# Combine results
all_trained_models.update(trained_models_eng)
all_comparisons_df = pd.concat([all_comparisons_df, comparison_df_eng], ignore_index=True)
all_comparisons_df = all_comparisons_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

all_comparisons_df


### 4.3.4 Conclusion: Engineered Features vs. Base Data

Surprisingly, adding the explicitly engineered Photometric Color Indices (`u-g`, `g-r`, etc.) resulted in a very slight performance *drop* (e.g., `eng_XGBoost` fell from 96.79% to 96.74%). 

**What this means:** Advanced tree-based algorithms like XGBoost and HistGradientBoosting are already inherently capable of finding the optimal splits and relationships between the raw photometric bands without us explicitly calculating the differences for them. Therefore, we will discard the engineered features and confidently proceed with the **base dataset** as our absolute best performer!

## 4.4 Hyperparameter Tuning with Optuna

Because we have run several heavy experiments (like SMOTE and feature engineering), our RAM is likely completely full, which causes `Out of Memory (OOM)` crashes. 

First, we will delete those unused datasets to free up memory. Then, we will run **Optuna** with strict memory management (`gc.collect()`) to safely squeeze out those last fractions of a percent in accuracy!

In [ ]:
# 1. Clean up memory from previous experiments to prevent OOM
for var in ['X_train_bal', 'y_train_bal', 'X_train_under', 'y_train_under', 'X_train_eng_raw', 'X_test_eng_raw', 'X_train_eng', 'X_test_eng']:
    if var in locals():
        del locals()[var]
gc.collect()

# 2. Define the objective function for Optuna
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400, step=100), # Capped to prevent OOM
        'max_depth': trial.suggest_int('max_depth', 4, 8), # Capped to prevent OOM
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'tree_method': 'hist',
        'device': GLOBAL_DEVICE,
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = XGBClassifier(**param)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    # Explicit garbage collection for each trial
    del model, y_pred
    gc.collect()
    
    return acc

# 3. Run Optimization
study = optuna.create_study(direction='maximize', study_name="XGBoost Tuning")
print("\n--- Starting Optuna Tuning for XGBoost ---")
study.optimize(objective, n_trials=10, gc_after_trial=True) # Force memory cleanup after each trial

print(f"\nBest Trial Accuracy: {study.best_value:.5f}")
print("Best Params:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

# 4. Train final tuned model and append to tracker
best_xgb_params = study.best_params
best_xgb_params.update({'tree_method': 'hist', 'device': GLOBAL_DEVICE, 'random_state': 42, 'n_jobs': -1})

tuned_xgb = XGBClassifier(**best_xgb_params)
tuned_xgb.fit(X_train, y_train)
all_trained_models['tuned_XGBoost'] = tuned_xgb

tuned_results = pd.DataFrame([{'Model': 'tuned_XGBoost', 'Accuracy': accuracy_score(y_test, tuned_xgb.predict(X_test))}])
all_comparisons_df = pd.concat([all_comparisons_df, tuned_results], ignore_index=True).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
all_comparisons_df


### 4.5 Error Analysis: Confusion Matrix

Let's visualize exactly where our best model is making its mistakes to see which stellar classes are the hardest to distinguish.

In [ ]:
best_model_name = all_comparisons_df.iloc[0]['Model']
best_model = all_trained_models[best_model_name]
y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['GALAXY', 'QSO', 'STAR'])
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title(f"Confusion Matrix for {best_model_name}")
plt.show()

# Print text output 
print("\n--- Confusion Matrix (Text) ---")
print(pd.DataFrame(cm, index=['True GALAXY', 'True QSO', 'True STAR'], columns=['Pred GALAXY', 'Pred QSO', 'Pred STAR']))



### 4.5.1 Conclusion: Confusion Matrix Insights"
    
Here is the breakdown of what the matrix tells us,

**1. The Model's Strengths**
*   **Galaxies are extremely well predicted:** Out of 75,694 actual Galaxies, the model correctly identified 74,050 of them (a **~97.8% recall** rate).
*   **Quasars (QSO) are also very strong:** Out of 23,243 actual Quasars, it correctly found 22,414 of them (**~96.4% recall**).
  
**2. The Main Weakness: STAR vs. GALAXY Confusion**
The absolute biggest source of error in your model is the overlap between Stars and Galaxies.
*   **1,097 true STARs** were misclassified as GALAXIES (this is a roughly **6.6% error rate** for the Star class, making it the model's weakest point).
*   Conversely, **975 true GALAXIES** were misclassified as STARs.
   
*Astrophysics Context:* This makes perfect sense! Photometrically (using just `u`, `g`, `r`, `i`, `z` light bands), distant, unresolved galaxies can appear point-like and mimic the light signatures of foreground stars within our own Milky Way. 

**3. The Secondary Weakness: QSO vs. GALAXY Confusion**
*   **603 true QSOs** were misclassified as GALAXIES.
*   **669 true GALAXIES** were misclassified as QSOs.
 
*Astrophysics Context:* Quasars (QSOs) are essentially the extraordinarily bright, active galactic nuclei (AGN) of distant galaxies. Because a Quasar is physically *inside* a host galaxy, the light we receive is sometimes a blend of both the Quasar and the galaxy, making it a tricky edge case for the model.

**4. The Least Confused: STAR vs. QSO**
*   Only **129 true STARs** were predicted as QSOs, and only **226 true QSOs** were predicted as STARs.
*   *Why?* Quasars (QSOs) and Stars have vastly different `redshift` values. Since `redshift` is your model's #2 most important feature, XGBoost easily uses it to draw a hard boundary between the two."


# 5.Submission

In [ ]:
# 1. Prepare test data
test_X = test_df.copy()
test_X = test_X.drop(columns=['id'])

# Map 'galaxy_population' the same as train_df (Red_Sequence -> 0, Blue_Cloud -> 1)
test_X['galaxy_population'] = test_X['galaxy_population'].map({'Red_Sequence': 0, 'Blue_Cloud': 1})

# One-hot encode 'spectral_type'
test_X = pd.get_dummies(test_X, columns=['spectral_type'], drop_first=True, dtype=int)

# Align column order with training data and fill any missing dummy columns with 0
test_X = test_X.reindex(columns=X_train.columns, fill_value=0)

# Apply the exact same scalers fitted on our training data to the submission test data
test_X = transform_numerical_features(test_X, training_scalers)

# Determine the overall best model from both training runs
best_model_name = all_comparisons_df.iloc[0]['Model']
best_model = all_trained_models[best_model_name]
print(f"Selected '{best_model_name}' as the best model for submission.\n")

# 2. Make predictions and map numerical predictions back to string labels
reverse_class_mapping = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}
submission_df['class'] = pd.Series(best_model.predict(test_X)).map(reverse_class_mapping)

# 3. Save submission file
submission_df.to_csv('submission.csv', index=False)
print("Submission saved successfully!")
submission_df.head()